# E3 — Transformed Müller–Brown (10D): a single-basin target with rare escapes

$$U(z) = \frac{1}{s}\,U_{\rm MB}(z_1,z_2) + \frac{1}{2\sigma_{\rm aux}^2}\sum_{\ell=3}^{10} z_\ell^2,\qquad s = 40,\ \sigma_{\rm aux} = 0.4,$$
sampled in mixed coordinates $x = z B^\top$, $B = Q\,\mathrm{diag}(\mathrm{linspace}(0.75,1.45,10))$ with $Q$ from QR of a `default_rng(12345)` normal; $V(x) = U(xB^{-\top})$ and $\nabla_x V = (\nabla_z U)B^{-1}$.

**Initialise in basin B.** The target occupancy is $\approx(0.9994,\ 0.0006,\ 10^{-5})$ for $(A, B, C)$ — Müller–Brown at low temperature **is a single-basin target; this is intrinsic to the potential, not a calibration failure**. The task is escaping the shallow initial basin $B$ and putting the mass where it belongs; EJS (bounded, quadratic near the target) is the primary occupancy metric because $p^\star$ is far from uniform and TV saturates.

In [ ]:
EXPERIMENT = "muller_brown_10d"
import os, sys, math, time, json
sys.path.insert(0, os.path.abspath(".."))
from src.gpu_guard import select_gpu
select_gpu(int(os.environ.get("JCP_GPU", "4")))
import torch
assert torch.cuda.device_count() == 1, "GPU guard must mask to exactly one device"
torch.set_default_dtype(torch.float64)
import numpy as np
import pandas as pd

from src import config as C
from src.experiments import build_e3, make_sampler_factory, make_metrics
from src.runner import (run_experiment, run_one, refine_dt, quadrature_refinement,
                        write_timeseries_csv, write_summary_csv, write_manifest,
                        ula_first_passage, hardware_manifest)
from src.samplers import tune_ladder
from src.certificate import make_phi_family, certificate_grid, certificate_importance
from src.plotting import make_all_figures, apply_style

DEV = "cuda"
RESULTS = os.path.abspath(os.path.join("..", "results", EXPERIMENT))
FIGURES = os.path.abspath(os.path.join("..", "figures", EXPERIMENT))
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
exp = build_e3(device=DEV, basin_cache=os.path.join(RESULTS, "basin_map.npz"))
cfg = exp.cfg
print(f"experiment={cfg.name}  d={cfg.d}  N={cfg.n_particles}  T={cfg.T}  dt0={cfg.dt}")
print(f"beta={cfg.beta}  eps={cfg.eps}  lambda={cfg.lam}  seeds={cfg.seeds}")
print(hardware_manifest())

## The model

$$U_{\rm MB}(\zeta) = \sum_{i=1}^4 A_i \exp\!\big[a_i(\zeta_1-x_i)^2 + b_i(\zeta_1-x_i)(\zeta_2-y_i) + c_i(\zeta_2-y_i)^2\big]$$
with $A=(-200,-100,-170,15)$, $a=(-1,-1,-6.5,0.7)$, $b=(0,0,11,0.6)$, $c=(-10,-10,-6.5,0.7)$, $x=(1,0,-0.5,-1)$, $y=(0,0.5,1.5,1)$. Verified critical points (asserted to 4 decimals below):

| | location | $U_{\rm MB}$ |
|---|---|---|
| min A | $(-0.5582,\ 1.4417)$ | $-146.70$ |
| min B | $(0.6235,\ 0.0280)$ | $-108.17$ |
| min C | $(-0.0500,\ 0.4667)$ | $-80.77$ |
| saddle S1 (A↔C) | $(-0.8220,\ 0.6243)$ | $-40.66$ |
| saddle S2 (C↔B) | $(0.2125,\ 0.2930)$ | $-72.25$ |

Connectivity is a **chain** $A \leftrightarrow C \leftrightarrow B$. Escape barrier from $B$: $35.92$, so $\beta b_B/s = 7.18$ and $\tau_{\rm Kramers}(B) \approx 940$.

Protocol: $N=2000$, $T=200$, $\Delta t_0=0.005$; box clipped in **latent** coordinates $z_1\in[-3,3]$, $z_2\in[-1.5,3.5]$, $z_{3:10}\in[-2,2]$ — deliberately generous, because the score integrand's support extends a full jump length beyond the target's effective support (do not shrink it). Metrics are primary in latent 2D $z_{1:2}=(xB^{-\top})_{1:2}$; full-10D sliced $W_2$ is also reported with its own bias floor (essential in 10D). Partition ($K=3$): gradient-flow basin map on a cached latent grid; $p^\star$ by grid quadrature.

In [ ]:
from src.potentials import MB_CRITICAL, muller_brown_2d, muller_brown_2d_grad, newton_refine
for key, (z_tab, U_tab) in MB_CRITICAL.items():
    z = newton_refine(muller_brown_2d_grad, torch.tensor(z_tab, device=DEV))
    U = muller_brown_2d(z.unsqueeze(0))[0].item()
    assert abs(z[0].item() - z_tab[0]) < 5e-5 and abs(z[1].item() - z_tab[1]) < 5e-5, key
    assert abs(U - U_tab) < 5e-2, (key, U)
    print(f"{key:>10s}: ({z[0].item():+.4f}, {z[1].item():+.4f})  U_MB = {U:9.4f}  [table: {z_tab}, {U_tab}]")
print(f"escape barrier from B = {exp.extras['barrier_B']*exp.pot.s:.2f}, "
      f"beta*b/s = {C.BETA*exp.extras['barrier_B']:.2f}, Kramers tau ~ {exp.kramers_tau:.0f}")
print("p_star (A, B, C):", np.round(exp.p_star.cpu().numpy(), 6),
      " -- single-basin target, intrinsic to Mueller-Brown at this temperature")

g = torch.Generator(device=DEV); g.manual_seed(0)
barrier_report = ula_first_passage(exp.pot, exp.box, exp.init_fn(cfg.n_particles, g),
                                   exp.exit_committed, cfg.dt, int(cfg.T/cfg.dt), C.EPS, g)
barrier_report["kramers_tau_B"] = exp.kramers_tau
print("ULA first-passage out of basin B:", barrier_report)

## Jump law and score

Euclidean MST on the three latent minima gives edges $(C,B)$ (length 0.804) and $(A,C)$ (length 1.10), symmetrised to 4 directed atoms $r_a = (\Delta z,\ 0_8)B^\top$, $w_a = \tfrac14$, shell $h = 0.1\,\min_a\|r_a\|$, $\lambda = 1$. Score: generic shell (log-space, as in E1) — the potential evaluations for $V(x-\theta_p r_{a,q})$ run over $Q_\theta\times A\times Q_\rho$ shifted copies, vectorised over the ensemble.

In [ ]:
print("latent minima:", {k: np.round(v.cpu().numpy(), 4).tolist()
                          for k, v in exp.extras["minima_latent"].items()})
print("MST atoms (latent dz):", np.round(exp.extras["atoms_z"][:, :2].cpu().numpy(), 4).tolist())
print("edge lengths:", [round(float(exp.extras['atoms_z'][i, :2].norm()), 4) for i in (1, 2)],
      " h (x-space):", round(exp.extras["h"], 4))

# certificate operates on the EXACT latent 2D reduction (jumps and test
# functions act on z_{1:2} only; dot products are affine-invariant; the aux
# Gaussian factorises) with per-atom shell widths carrying h into z-space
from src.potentials import MuellerBrownLatent2D
from src.jumps import ShellJumpLaw
from src.score import ShellScore
potr = MuellerBrownLatent2D(s=exp.pot.s)
dz = exp.extras["atoms_z"][:, :2]
atoms_x = exp.law.atoms
h_z = exp.extras["h"] * dz.norm(dim=1) / atoms_x.norm(dim=1)
law_r = ShellJumpLaw(dz, torch.full((4,), 0.25, device=DEV), h_z)
DEFAULT_QUAD = dict(q_theta=C.Q_THETA, q_rho=C.Q_RHO)
phis = make_phi_family(2, [0.0, 0.8], 0.8, DEV)

def cert_e3(q_theta, q_rho):
    score = ShellScore(potr, law_r, cfg.lam, cfg.beta, q_theta, q_rho)
    shifts, logw = law_r.quadrature_shifts(64)   # fine continuous-nu J side
    return certificate_grid(potr, score, shifts, logw, cfg.lam, cfg.beta, phis,
                            [-4.2, -2.7], [4.2, 4.7],
                            n_panels=130, nodes_per_panel=8, chunk=8192)

## Target preservation: the stationarity identity $(\star)$

The LSC-CP generator is
$$\mathcal A f = \big[-\nabla V + S_{\nu,\beta}\big]\cdot\nabla f + \varepsilon\,\Delta f + \lambda\!\int\!\big[f(x+r)-f(x)\big]\nu(dr),$$
with $\nu$ a **probability** measure and
$$S_{\nu,\beta}(x) = -\lambda \int \nu(dr)\; r \int_0^1 \exp\!\Big[-\beta\big(V(x-\theta r) - V(x)\big)\Big]\, d\theta .$$
Raw CP is the same generator with $S \equiv 0$.

Write $p = e^{-\beta V}/Z$. The overdamped part is $\pi$-reversible, so invariance of $\pi$ is equivalent to
$$\int S\cdot\nabla\varphi \, d\pi + \int J\varphi \, d\pi = 0 \qquad \forall\, \varphi \in C_c^\infty . \tag{$\star$}$$

**Jump term.** Shift the integration variable and apply the fundamental theorem of calculus along $\theta \mapsto y - \theta r$:
$$\int J\varphi\,d\pi = \lambda\!\int\!\nu(dr)\!\int\!\varphi(y)\big[p(y-r)-p(y)\big]dy = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(y)\,r\!\cdot\!\nabla p(y-\theta r)\,dy\,d\theta.$$

**Drift term.** $S(x) = -\lambda\int\nu(dr)\,r\int_0^1 \frac{p(x-\theta r)}{p(x)}d\theta$ (identical to the boxed formula since $p \propto e^{-\beta V}$), so integrating by parts in $x$:
$$\int S\cdot\nabla\varphi\,d\pi = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\big(r\!\cdot\!\nabla\varphi(x)\big) p(x-\theta r)\,dx\,d\theta = +\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(x)\,r\!\cdot\!\nabla p(x-\theta r)\,dx\,d\theta.$$

The two cancel identically. **Target preservation is unconditional in $\nu$** — any finite-activity jump law works; only the *speed* depends on $\nu$.

### The measured certificate $\mathcal R(\varphi)$

For smooth bounded test functions (products of tanh ridges) we report
$$\mathcal R(\varphi) = \frac{\big|\int S_{\nu,\beta}\!\cdot\!\nabla\varphi\,d\pi + \int J_\nu\varphi\,d\pi\big|}{\big|\int J_\nu\varphi\,d\pi\big|},$$
zero in exact arithmetic; the measured value is the combined defect of the $\theta$/$\rho$ quadratures. Two implementation notes, both load-bearing:

1. **The integration domain extends a full jump length beyond the target's effective support.** Order-one contributions to $(\star)$ live where $\pi$ is tiny and $S$ is enormous; a deliberately tight box produces a large residual (demonstrated below and regression-tested).
2. **The drift integrand $p\,S\cdot\nabla\varphi$ is assembled in log space** from the score's $(M, v)$ parts as $\exp(-\beta V + M)\,v\cdot\nabla\varphi$: in linear fp64 arithmetic $p$ underflows exactly where $\|S\|$ is astronomical, silently dropping those order-one far-field contributions. The residual uses the *uncapped* $M$; the deployed drift caps $M$ at $M_{\max}=600$, but because taming saturates (the tamed step tends to $-v/\|v\|$), the deployed tamed step differs from the uncapped one by $O(e^{-M_{\max}})$ — that saturation defect is reported alongside $\mathcal R$ and is $\lesssim 10^{-250}$ here.

A useful exact identity (change of variables $x \to x+\theta_p r$ in the drift term): for the *implemented* quadrature score,
$$\int S\cdot\nabla\varphi\,d\pi + \int J\varphi\,d\pi \;=\; \lambda\,\mathbb E_\pi\!\int\!\nu(dr)\Big[\varphi(x+r)-\varphi(x) - \sum_p \hat w_p\, r\cdot\nabla\varphi(x+\theta_p r)\Big],$$
i.e. the residual is **independent of $V$** and equals the $\theta$-quadrature error on the smooth test-function integrand. This is why moderate pointwise errors of the Gauss–Legendre rule on the stiff factor $e^{\beta\Delta V}$ do not translate into a weak (distributional) defect of the sampled law.

In [ ]:
cert_report = cert_e3(**DEFAULT_QUAD)
print("R(phi), latent-2D reduction, generous latent box (metric box + jump reach):")
for i in range(len(phis)):
    print(f"  phi_{i}: R = {cert_report[f'phi_{i}']['residual']:.3e}")
print(f"max R = {cert_report['max_residual']:.3e}  "
      f"clip saturation defect = {cert_report['clip_tamed_step_defect']:.2e}")
assert cert_report["max_residual"] < 1e-6

## The seven methods

All methods share one **taming policy**: the same map $b \mapsto b/(1+\Delta t\,\|b\|)$ is applied to every method's drift (ULA, the MALA proposal, FLA, the BAOAB force, raw CP, LSC-CP). Tamed MALA is still exact because the proposal density $q(y|x) = \mathcal N(y;\, x + \Delta t\, b_{\rm tamed}(x),\, 2\varepsilon\Delta t\, I)$ is used consistently in both directions of the MH ratio; asymmetric taming would make taming a hidden variable in the comparison.

**1. ULA.** $X \leftarrow X + \Delta t\,\mathrm{tame}(-\nabla V) + \sqrt{2\varepsilon\Delta t}\,\xi$.

**2. MALA.** With $\nabla\log\pi = -\beta\nabla V$, the proposal $Y = X - \tfrac{h\beta}2\nabla V(X) + \sqrt h\,\xi$ matches the ULA step iff $\tfrac{h\beta}{2} = \Delta t$ **and** $h = 2\varepsilon\Delta t$. Both conditions coincide:
$$\boxed{h = \frac{2\Delta t}{\beta} = 2\varepsilon\Delta t = \Delta t/4 \quad\text{at }\beta=8.}$$
Log-acceptance
$$\log\alpha = -\beta[V(Y)-V(X)] - \frac{1}{2h}\Big[\|X - \mu(Y)\|^2 - \|Y-\mu(X)\|^2\Big],\qquad \mu(z) = z + \Delta t\,\mathrm{tame}(-\nabla V(z)),$$
accepted elementwise. Proposals are **never clipped before the accept step** (that silently breaks exactness); out-of-box proposals are auto-rejected, which is valid MH for the box-restricted target. Expect acceptance $\approx 1$ — the honest message: **rejection does not cure metastability**.

**3. FLA / FLMC** (Şimşekli, ICML 2017, §3.3). $X \leftarrow X + \Delta t\,\mathrm{tame}(-c_\alpha \nabla U) + \Delta t^{1/\alpha}\,\xi^{(\alpha)}$ with $U = \beta V$, $c_\alpha = \Gamma(\alpha-1)/\Gamma(\alpha/2)^2$, $\alpha = 1.7$; per-coordinate $S\alpha S(1)$ noise by Chambers–Mallows–Stuck. **No tail clipping** — a truncated stable is not stable. FLA is the *uncorrected nonlocal* comparator: heavy tails cross barriers, but the invariant law is not $\pi$.

**4. Kinetic Langevin (BAOAB).** It is **not HMC** (no accept/reject; carries $O(\Delta t^2)$ configurational bias). Unit mass, $\gamma = 1$; the O-step coefficient is the exact OU solution: $dp = -\gamma p\,dt + \sqrt{2\gamma\varepsilon}\,dW$ gives $\mathrm{Var} = 2\gamma\varepsilon\int_0^{\Delta t}e^{-2\gamma s}ds = \varepsilon(1-e^{-2\gamma\Delta t})$ by Itô isometry. The trailing force is cached as the next step's leading B (one gradient per step).

**5. Parallel tempering.** MALA-within-replica at $\beta_k = \beta\, r^{k-1}$, replica $k$ using $h_k = 2\Delta t/\beta_k$ (same tamed drift step for every replica; only the noise scale differs). Adjacent swaps every $n_{\rm swap}$ steps (alternating parity); the joint target is $\prod_k \pi_k$ and the swap is a deterministic involution, so
$$\alpha_{\rm swap} = \min\Big\{1,\ \exp\big[(\beta_i - \beta_{i+1})\big(V(x_i) - V(x_{i+1})\big)\big]\Big\}.$$
$V$ values are cached by MALA, so swaps are free in evaluation count. $K$ is tuned so the mean swap acceptance lands in $[0.2, 0.4]$. The replica index is a batch dimension, state $(K, N, d)$; metrics use the cold replica only; **wall-clock includes all $K$ replicas**.

**6/7. Raw CP and LSC-CP.** Identical time discretisation,
$$X^{(1)} = X_n + \Delta t\,\frac{b(X_n)}{1+\Delta t\,\|b(X_n)\|} + \sqrt{2\varepsilon\Delta t}\;\xi_n,\qquad X_{n+1} = X^{(1)} + \sum_{k=1}^{N_n} A_k,$$
$N_n \sim \mathrm{Poisson}(\lambda\Delta t)$, $A_k \stackrel{iid}\sim \nu$; $b = -\nabla V$ (raw CP) or $b = -\nabla V + S_{\nu,\beta}$ (LSC-CP). The jump stream is a dedicated generator seeded identically for both methods, so their jump times and increments are **pathwise identical** (verified in `tests/test_samplers.py`), not merely equal in law.

In [ ]:
# PT ladder: geometric in beta, K tuned so mean swap acceptance is in [0.2, 0.4]
gen = torch.Generator(device=DEV); gen.manual_seed(0)
x0_pilot = exp.init_fn(min(512, cfg.n_particles), gen)
pt_betas, ladder_info = tune_ladder(exp.pot, x0_pilot, cfg.dt, exp.box,
                                    C.BETA, exp.pt_beta_min, pilot_steps=600)
print(f"PT ladder: K={ladder_info['K']}  r={ladder_info['r']:.4f}  "
      f"beta_K={pt_betas[-1].item():.4f}  swap acceptance={ladder_info['swap_acceptance']:.3f}")
print("tuning history {K: acceptance}:", ladder_info["history"])

## Reference, partition, metrics, bias floors

Reference sample size equals the run's $N$; metrics are evaluated at every checkpoint (cadence fixed in $t$, identical across methods).

* **$W_2$**: exact in 1D (sorted coupling); **sliced** $W_2$ for $d\ge2$ with $L=200$ projections drawn once from a fixed seed and reused across all times and methods (its bias floor decays like $N^{-1/2}$, not $N^{-1/d}$).
* **TV** (occupancy, on the partition): $\tfrac12\sum_k|\hat p_k - p^\star_k|$ — a **lower bound** on the full TV.
* **MMD**: Gaussian kernel, bandwidth **frozen once** by the median heuristic on the reference sample (per-frame bandwidths would make curves non-comparable); biased V-statistic $\widehat{\mathrm{MMD}}_b^2 = \|\mu_X-\mu_Y\|_{\mathcal H}^2 \ge 0$.
* **EMC** $= e^{H(\hat p)}/K$: plotted with a horizontal line at the target $e^{H(p^\star)}/K$; EMC $=1$ is optimal only for uniform $p^\star$, and deviation in *either* direction is error.
* **EJS**: base-2 Jensen–Shannon divergence between $\hat p$ and $p^\star$ (Blessing et al., arXiv:2406.07423, App. A.3), bounded in $[0,1]$, quadratic near the target, so it stays informative where TV saturates.
* **Bias floors** (mandatory): each metric between two independent reference samples of size $N$, 20 replicates; dashed line on every panel. Without this, every plateau is uninterpretable.
* **Nonfinite fraction**: logged per method per checkpoint; must be identically zero — metrics on survivors only would be survivorship bias, so nothing is ever filtered.

**Coverage vs correctness, once:** EMC measures *coverage*, TV/EJS measure *correctness*. Raw CP, whose invariant law is not $\pi$, over-flattens — driving EMC toward 1 (above its target line) while TV and EJS stay bad. That pairing *is* the raw-CP-vs-LSC-CP story.

In [ ]:
metrics_fn, floors, aux = make_metrics(exp, cfg.n_particles)
emc_target = exp.emc_target
print("p_star:", np.round(exp.p_star.cpu().numpy(), 6))
print("EMC target line: %.4f" % emc_target)
print("MMD bandwidth (median heuristic on reference, frozen):", round(aux["bandwidth"], 4))
print("bias floors (mean +- std over 20 replicate pairs):")
for k, v in floors.items():
    print(f"  {k:>12s}: {v['mean']:.5f} +- {v['std']:.5f}")

## $\Delta t$ refinement and production

Declared $\Delta t$ selection rule, applied uniformly to every experiment (reported in the SI): **the largest $\Delta t$ on a dyadic grid at which every method's terminal value of every metric is within 5% of its $\Delta t/2$ value.** Three statistical guards make the rule meaningful at a single refinement seed: differences are measured relative to $\max(|m_{\Delta t/2}|,\ \text{bias floor})$; when *both* values sit inside the floor band (floor mean $+\,3$ s.d.) they are declared in agreement; and differences within $4\times$ the floor s.d. — the natural unit of single-run metric sampling noise at this $N$ — are likewise noise, not discretisation bias. The same guards apply to the quadrature-refinement comparison.

**One declared exception:** the two comparators whose invariant law is not $\pi$ — FLA and raw CP — do not gate the $\Delta t$ selection. Their terminal values measure an intrinsic bias, not convergence to a target, so there is no $\Delta t$ at which they *should* stabilise to 5% at single-seed resolution (empirically FLA's density error drifts monotonically under refinement, and raw CP's bias wobbles at its own sampling noise); demanding stability from them would refine $\Delta t$ forever. Both still run at the shared chosen $\Delta t$, and their deviations across the dyadic grid are recorded in the refinement table for transparency. The gate is carried by the five $\pi$-targeting methods.

Production protocol: 5 seeds $\times$ 7 methods, run **sequentially** (never batched) so per-run wall-clock is meaningful; all methods share $x_0$ per seed; 20 untimed warm-up steps absorb allocator/JIT effects; `torch.cuda.synchronize()` brackets every timed region, and the timer covers sampler work only.

In [ ]:
def run_terminal_lsc(**quad):
    f = make_sampler_factory(exp, cfg.dt, pt_betas, score_kwargs=quad)
    n_ = int(round(cfg.T / cfg.dt))
    r_, _ = run_one("LSC-CP", 0, f, n_, n_, cfg.dt, metrics_fn, exp.pot, quiet=True)
    return {k: r_[-1][k] for k in ("W2", "TV", "MMD", "EMC", "EJS", "W2_10d")}

settings = [dict(q_theta=qt, q_rho=qr) for qt in (8, 16, 32) for qr in (4, 8, 16)]
CHOSEN_QUAD, quad_table = quadrature_refinement(
    settings, run_terminal_lsc, lambda **s: cert_e3(**s)["max_residual"], floors)
print("chosen production quadrature:", CHOSEN_QUAD)
display(pd.DataFrame(quad_table).round(6))
if CHOSEN_QUAD != DEFAULT_QUAD:
    cert_report = cert_e3(**CHOSEN_QUAD)
    print("certificate re-evaluated at chosen orders: max R =",
          f"{cert_report['max_residual']:.3e}")
    assert cert_report["max_residual"] < 1e-6

### E3 exception: $\Delta t$ is declared, not certified

On this landscape the dyadic rule cannot certify any practical $\Delta t$ for LSC-CP, and the failure is *mechanistic, not statistical* (diagnosed in the mechanism section below): uphill jump-landers receive a corrective drift of magnitude $e^{\beta\Delta V}$ (up to $e^{21}$ here — the only experiment with strongly asymmetric basin depths), and the tamed Euler step turns the corresponding smooth return flow into a single $O(1)$ hop whose landing scatter parks a $\Delta t$-independent fraction of mass in score-dark regions. Since the tamed step saturates for any $\Delta t > e^{-M}$, refining $\Delta t$ cannot converge these terminal values. Production therefore runs at the declared $\Delta t_0$ (well inside the stability bound), the level-0 comparison table is recorded, and `dt_certified = False` is written to the manifest. LSC-CP's terminal metrics carry a documented $\Delta t$-sensitivity of the order of the table's spread.

In [ ]:
MAIN_METRICS = ["W2", "TV", "MMD", "EMC", "EJS", "W2_10d"]

def run_terminal_all(dt_):
    n_ = int(round(cfg.T / dt_))
    factory = make_sampler_factory(exp, dt_, pt_betas, score_kwargs=CHOSEN_QUAD)
    out = {}
    for m in C.METHODS:
        rows_, _ = run_one(m, 0, factory, n_, n_, dt_, metrics_fn, exp.pot, quiet=True)
        out[m] = {k: rows_[-1][k] for k in MAIN_METRICS}
    print(f"  refine_dt: finished pass at dt={dt_}", flush=True)
    return out

_, dt_table = refine_dt(run_terminal_all, cfg.dt, floors, exclude=("FLA", "CP"),
                        max_halvings=1)
dt_certified = bool(dt_table[0]["pass"])
for row in dt_table:
    print(row)
dt_final = cfg.dt          # declared (see markdown above); certification recorded
print(f"dt_certified = {dt_certified}; production at declared dt0 = {dt_final}")

n_steps = int(round(cfg.T / dt_final))
steps_per_ck = max(1, n_steps // C.N_CHECKPOINTS)
factory = make_sampler_factory(exp, dt_final, pt_betas, score_kwargs=CHOSEN_QUAD)
t0 = time.time()
rows, method_info = run_experiment(C.METHODS, cfg.seeds, factory, n_steps,
                                   steps_per_ck, dt_final, metrics_fn, exp.pot)
print(f"production total: {time.time()-t0:.0f}s")
worst_nonfinite = max(r["nonfinite_frac"] for r in rows)
assert worst_nonfinite == 0.0, worst_nonfinite
print("nonfinite fraction: identically zero across all methods/checkpoints")

## Mechanism: why the corrected process stalls on a quasi-stationary plateau

The MST law's uphill jumps (into $C$, whose equilibrium mass is $\sim 5\times10^{-6}$) land with an enormous, *correctly aimed* Lévy score — the correction implementing detailed-balance rejection as a drift. Three measurements pin the failure of the tamed discretisation to represent that drift:

1. **Occupancy trajectories**: LSC-CP reaches its plateau by $t\approx25$ and then stalls (it is quasi-stationary, not slowly relaxing), and the plateau *worsens* slightly at finer $\Delta t$; raw CP is $\Delta t$-stable at a much worse law; locals barely leave B.
2. **Lander cohort**: an equilibrated A-ensemble jumped by the exact $A\to C$ atom lands 99% in the C label with median score log-magnitude $M\approx 8$; one tamed step returns most of it, but the single $O(1)$ hop scatters — after a few steps a $\Delta t$-independent $\sim$17% is parked with $M\approx-3$ (score-dark), rescued only by later jumps.
3. **Score field**: $\|S\|\approx 5\times10^{3}$ at $z_C$ pointing along $C\to A$ (correct); $O(10^{-3})$ at the inhabited minima. The stiffness is confined to the jump shadows — exactly where landers appear.

E1/E2/E4 are immune because their jump laws connect (near-)iso-energetic minima, so scores at inhabited points are $O(1)$. The finding: **on landscapes with strongly asymmetric basin depths, the exactness of the Lévy-score correction concentrates in $e^{\beta\Delta V}$-stiff drifts that the tamed Euler scheme cannot integrate**; a discretisation that resolves the post-jump return flow (or a Metropolised jump step) is required to realise the generator's exactness in practice.

In [ ]:
# 1) occupancy trajectories: quasi-stationary plateau, dt-dependence
from src.metrics import occupancy as _occ

def occ_traj(sampler, n_steps_, every, label):
    out = []
    for s_ in range(n_steps_):
        sampler.step()
        if (s_ + 1) % every == 0:
            p = _occ(exp.labels_fn(sampler.positions()), 3)
            out.append(((s_ + 1) * sampler.dt, [round(float(v), 4) for v in p]))
    print(label)
    for t_, p in out:
        print(f"   t={t_:6.1f}: A={p[0]:.4f} B={p[1]:.4f} C={p[2]:.4f}")
    return out

mech_traj = {}
for meth, dt_ in (("LSC-CP", dt_final), ("LSC-CP", dt_final / 4.0),
                  ("CP", dt_final), ("ULA", dt_final)):
    f = make_sampler_factory(exp, dt_, pt_betas, score_kwargs=CHOSEN_QUAD)
    s = f(meth, 0)
    n_ = int(round(cfg.T / dt_))
    mech_traj[f"{meth}@dt={dt_}"] = occ_traj(s, n_, n_ // 8, f"{meth} dt={dt_}")

In [ ]:
# 2) lander cohort: single-hop return + dt-independent parked fraction
from src.samplers import tame as _tame
score_m = exp.make_score(**CHOSEN_QUAD)
zA_, zC_ = exp.extras["minima_latent"]["min_A"], exp.extras["minima_latent"]["min_C"]
mechanism_report = {"traj": {k: v[-1][1] for k, v in mech_traj.items()}}
for dt_ in (dt_final, dt_final / 4.0):
    g_ = torch.Generator(device=DEV); g_.manual_seed(3)
    zloc = torch.zeros(4000, 10, device=DEV)
    zloc[:, :2] = zA_ + 0.12 * torch.randn(4000, 2, generator=g_, device=DEV)
    zloc[:, 2:] = 0.1414 * torch.randn(4000, 8, generator=g_, device=DEV)
    x_ = exp.pot.from_latent(zloc) + exp.law.atoms[2]      # jump A -> C
    M0, _ = score_m.log_parts(x_)
    for _s in range(24):
        S_, _d = score_m(x_)
        b_ = -exp.pot.grad(x_) + S_
        xi_ = torch.randn(x_.shape, generator=g_, device=DEV)
        x_ = exp.box.clip(x_ + dt_ * _tame(b_, dt_) + (2 * C.EPS * dt_) ** 0.5 * xi_)
    Mf, _ = score_m.log_parts(x_)
    occ_f = _occ(exp.labels_fn(x_), 3)
    parked = float(occ_f[2].item())
    print(f"dt={dt_}: lander median M {M0.median():.2f} -> after 24 steps "
          f"M {Mf.median():.2f}; parked C fraction {parked:.3f}")
    mechanism_report[f"parked_fraction@dt={dt_}"] = parked
S_min, _ = score_m.log_parts(exp.pot.from_latent(torch.cat([
    torch.cat([zA_, torch.zeros(8, device=DEV)]).unsqueeze(0),
    torch.cat([zC_, torch.zeros(8, device=DEV)]).unsqueeze(0)])))
mechanism_report["score_logmag_at_A_C"] = [float(v) for v in S_min]
print("score log-magnitude at (A, C):", mechanism_report["score_logmag_at_A_C"])

## Figures

In [ ]:
fig_metrics = ("W2", "TV", "MMD", "EMC", "EJS", "W2_10d")
written = make_all_figures(rows, FIGURES, floors, emc_target, metrics=fig_metrics)
print(f"{len(written)} figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> {FIGURES}")

# grid display for inspection (saved files above are one-figure-per-file)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
n_m = len(fig_metrics)
fig, axes = plt.subplots(n_m, 2, figsize=(11, 3.2 * n_m))
for i, metric in enumerate(fig_metrics):
    for j, tag in enumerate(("vs_time", "vs_wallclock")):
        ax = axes[i, j] if n_m > 1 else axes[j]
        ax.imshow(mpimg.imread(os.path.join(FIGURES, f"{metric}_{tag}.png")))
        ax.set_axis_off()
plt.tight_layout(); plt.show()

## CSV emission and summary

In [ ]:
ts_path = os.path.join(RESULTS, "metrics_timeseries.csv")
write_timeseries_csv(rows, ts_path)
summary_metrics = MAIN_METRICS + ["nonfinite_frac"]
summary = write_summary_csv(rows, C.METHODS, cfg.seeds, summary_metrics,
                            method_info, floors, os.path.join(RESULTS, "summary.csv"))

manifest = dict(
    experiment=EXPERIMENT,
    config=dict(d=cfg.d, N=cfg.n_particles, T=cfg.T, dt0=cfg.dt, dt=dt_final,
                beta=cfg.beta, eps=cfg.eps, lam=cfg.lam, seeds=list(cfg.seeds),
                n_checkpoints=C.N_CHECKPOINTS, warmup_steps=C.N_WARMUP_STEPS),
    quadrature=dict(chosen=CHOSEN_QUAD, table=quad_table),
    dt_refinement=[{k: (str(v) if isinstance(v, tuple) else v) for k, v in row.items()}
                   for row in dt_table],
    pt_ladder={k: v for k, v in ladder_info.items()},
    certificate=cert_report,
    bias_floors=floors,
    barrier_verification=barrier_report,
    method_info={m: {k: v for k, v in mi.items() if isinstance(v, (int, float))}
                 for m, mi in method_info.items()},
    hardware=hardware_manifest(),
    dt_certified=dt_certified, mechanism=mechanism_report,
)
write_manifest(os.path.join(RESULTS, "manifest.json"), **manifest)
print("wrote", ts_path)
from IPython.display import display
display(pd.read_csv(os.path.join(RESULTS, "summary.csv")).round(5))